In [48]:
# %pip install -U langchain-community faiss-cpu langchain-huggingface pymupdf tiktoken langchain-ollama python-dotenv

In [49]:
import os
import warnings
from dotenv import load_dotenv
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
warnings.filterwarnings("ignore" )
load_dotenv()

True

In [50]:
os.environ['LANGCHAIN_PROJECT']

'chatMyPdf'

In [51]:
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader('./rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf')
docs = loader.load()

In [52]:
docs
doc = docs[0]
# print(doc.page_content)

In [53]:
import os


pdfs=[]
for root, dirs, files in os.walk('rag-dataset'):
    # print(root,dirs,files)
    for file in files:
        if file.endswith(".pdf"):
            pdfs.append(os.path.join(root, file))

In [ ]:
pdfs

In [55]:
docs = []

for pdf in pdfs:
    loader = PyMuPDFLoader(pdf)
    pages = loader.load()

    docs.extend(pages)

In [56]:
# docs
print(len(docs))

64


In [57]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

In [58]:
print(len(docs),len(chunks))

64 311


In [59]:
# print(docs[0].page_content)
# print(chunks[0].page_content)

In [60]:
import tiktoken
encoding = tiktoken.encoding_for_model('gpt-4o-mini')
len(encoding.encode((docs[0].page_content))), len(encoding.encode((chunks[0].page_content)))

(968, 294)

In [61]:
from langchain_ollama import OllamaEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import  InMemoryDocstore

In [62]:
embeddings = OllamaEmbeddings(model='nomic-embed-text',
                              base_url='http://localhost:11434')
single_vector = embeddings.embed_query('This is some text data')

In [63]:
len(single_vector)

768

In [64]:
index = faiss.IndexFlatL2(len(single_vector))
index.ntotal,index.d

(0, 768)

In [65]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [66]:
vector_store

In [67]:
ids = vector_store.add_documents(documents=chunks)
len(ids)

311

In [ ]:
vector_store.index_to_docstore_id
len(vector_store.index_to_docstore_id)

In [ ]:
db_name = 'health_supplements'
vector_store.save_local(db_name)

In [ ]:
new_vs = FAISS.load_local(db_name,embeddings=embeddings,
                          allow_dangerous_deserialization=True)

In [ ]:
print(len(new_vs.index_to_docstore_id))

In [73]:
# for doc in docs:
#     print(doc.page_content)
#     print('\n\n')

In [78]:
retriever = vector_store.as_retriever(search_type="mmr",
                                      search_kwargs= {
                                          'k':3,
                                          'fetch_k':100,
                                          'lambda_mult':1
                                      })

In [ ]:
# docs = retriever.invoke(ques[2])

# docs = retriever.invoke(ques)


In [ ]:
# for doc in docs:
#     print(doc.page_content,'\n\n')

In [89]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

In [90]:
model = ChatOllama(model = 'llama2',
                   base_url='http://localhost:11434')
# model.invoke('hi')

In [91]:
prompt  = hub.pull('rlm/rag-prompt')

In [85]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [ ]:
def format_docs(d):
    return "\n\n".join([doc.page_content for d in docs])
# print(format_docs(docs))

In [95]:
rag_chain = (
    {"context":retriever|format_docs,'question':RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [96]:
ques = ['what is used to gain muscle mass?',
        'what are the benifits of BCAA supplements?',
        'What are the side effects of supplements?',
        'what are the benifits of supplements?',
        'What is used to reduce weight?',
        ]
# docs=vector_store.search(query=ques,
# search_type='similarity'
# )

In [97]:
output = rag_chain.invoke(ques[2])

In [98]:
print(output)

Based on the context provided, the potential side effects of supplements include:

1. Vitamin and mineral supplements: While supplementing with these nutrients may be beneficial for individuals with deficiencies, taking excessive quantities can lead to detrimental side effects, especially when taken in combination with other supplements.
2. Omega-3 fatty acid supplements: While these supplements have been shown to be beneficial in treating lipid disorders, they may not be without risk. Case reports suggest that omega-3 supplements can cause adverse effects such as bleeding and muscle weakness.
3. Soy protein and plant-derived antioxidant and anti-inflammatory nutraceuticals: These supplements have been associated with more severe adverse effects, including allergic reactions and interactions with medications.
4. Weight-loss and body building supplements: These supplements can lead to adverse effects such as cardiovascular problems, liver damage, and kidney damage when taken in excess o